In [0]:
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

In [0]:
import json

from pyspark.sql import functions as F
from common_utils.logging import get_logger
from common_utils.goldfunctions import date_key, write_gold

# ============================================================
# 1. LOGGER
# ============================================================
logger = get_logger("s2g_dimcancellationreason")

# ============================================================
# 2. CONFIG PATH
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

logger.info("Reading Gold configuration from %s", config_path)

# ============================================================
# 3. READ CONFIG
# ============================================================
with open(config_path, "r") as file:
    config = json.load(file)


# ============================================================
# 4. CONFIG SECTIONS
# ============================================================
table_config = config["dim_cancellation_reason"]

source_table = table_config["source_table"]
target_table = table_config["target_table"]
key_column   = table_config["key_column"]

# ============================================================
# 5. READ SILVER TABLE
# ============================================================
logger.info("Reading Silver table: %s", source_table)

df = spark.table(source_table).drop("last_updated_timestamp")

# ============================================================
# 6. CHECK KEY IS UNIQUE
# ============================================================
if df.count() != df.select(key_column).distinct().count():
    raise ValueError(f"Duplicate {key_column} values found in {source_table}")

# ============================================================
# 7. DISPLAY DATA
# ============================================================
display(df)

# ============================================================
# 8. WRITE GOLD TABLE
# ============================================================
write_gold(spark, df, target_table)

# ============================================================
# 9. COMPLETED
# ============================================================
logger.info("Gold dim_cancellation_reason processing completed")